# 07 — One pixel of the current season: curve, crop dates and 5-3-2 chips

Set `AOI` and `PID`, then **Run All**. You get:

1. **Info** — location, how many dates were kept by the QA60 mask, the longest gap, last season's map class.
2. **Crop cycles** — transplanting (field start), green-up onset, peak, harvest, measured on the smoothed curve.
3. **The curve** — every raw date (kept / SCL-flagged / removed by QA60), the 5-day composite, the fitted NDVI and LSWI, with the crop dates as dashed lines and ▲ where a chip is shown.
4. **5-3-2 chips** — every date on which the pixel is clear (or the clearest per month), the pixel in the magenta box, chips near a crop date framed in that date's colour.

**Before the first run** the AOI's series must exist: `nd.build(<aoi>)` (see `docs/09`, section 2.5). Use the **sar-rice-mapper (.venv)** kernel. A pixel id comes from `<aoi>_pixel_index.tif` (QGIS *Identify*).

**Reading the dates:** `transplant_date` is the field's start (transplanting or direct seeding), estimated as the green-up onset minus 15 days — at transplanting the pixel is mostly water with small seedlings, so NDVI only climbs once they have tillered. `trough_date` is the detector's own lowest point; when the two are far apart the field sat low and flat for weeks before the crop.

## 1. Choose the pixel

In [ ]:
AOI = 143          # AOI number
PID = 5390         # pixel id from <aoi>_pixel_index.tif
HALF = 30          # chip half-width in pixels: 30 -> 61 x 61 px, about 610 m across
SELECT = "pixel_clear"  # "pixel_clear": every date the pixel itself is clear; "per_month": clearest PER_MONTH per month
PER_MONTH = 2      # used only with SELECT = "per_month"
STRETCH = "chip"   # "chip": sharpest contrast per chip; "series": one stretch, colours comparable across dates
GAMMA = 0.7        # below 1 brightens dark chips (dense canopy); 1.0 = no change
SAVE_TO = None     # e.g. "processed/_batch/s2_2026/aoi143/figures" to also save the two PNGs

## 2. Setup

Reloads the modules **in dependency order**: reloading only the last one would keep an old copy of the ones behind it.

In [ ]:
import importlib
import os
from pathlib import Path

import pandas as pd
from IPython.display import display

# run from the repository root, whichever folder the notebook was opened from
os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()))

from sar_pipeline.analysis import optical_phenology, ndvi_5day, pixel_2026
for module in (optical_phenology, ndvi_5day, pixel_2026):
    importlib.reload(module)
px = pixel_2026
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 250)

## 3. Build the report

In [ ]:
r = px.report(AOI, PID, half=HALF, per_month=PER_MONTH, stretch=STRETCH, gamma=GAMMA, select=SELECT, out_dir=SAVE_TO)
display(r["info"].to_frame())

## 4. Crop cycles

`complete = False` means the cycle runs into the start or end of the series (for example a crop still standing on the last date). Lengths are in days.

In [ ]:
display(r["cycles"])

## 5. The curve

In [ ]:
r["curve"]

## 6. 5-3-2 chips

By default every date on which the pixel itself is clear (Cloud Score+ >= 60 at the pixel).

In 5-3-2 a paddy goes: dark or bare at transplanting (black where water stands), light then dark as the canopy fills, yellow-brown at maturity, then bare after harvest.

In [ ]:
r["chips"]

## 7. Every date and how clear its chip was (optional)

In [ ]:
display(r["chip_dates"])